# 2024 유로 스페인 빌드업 패턴: possession 체인 추적

2024 UEFA 유로에서 스페인이 치른 7경기 전체에서, 같은 possession 안에서 연속된 성공 패스(빌드업 체인)만 걸러 구역 전진 경로를 그리고, 왼쪽/오른쪽으로 끝난 가장 긴 체인을 사례로 확인합니다.

- `plot_zone_progression()`(구역 기반 전진 경로)은 possession 구분 없이 매치 전체의 전진 패스를 모두 집계하는 반면, 이 노트북은 "같은 possession 안에서 팀의 성공 패스가 2회 이상 연속으로 이어진" 진짜 빌드업 체인의 패스만 사용합니다.
- 메인 패널은 `plot_zone_progression()`과 동일한 30구역 점유 히트맵 + 전진 화살표 스타일이고, 오른쪽에 왼쪽/오른쪽으로 끝난 가장 긴 체인 1개씩을 실제 좌표 그대로 이은 미니 패널 2개를 추가로 보여줍니다.
- 이 폴더(`possession_chains/`)는 "2024 유로 스페인의 빌드업 패턴" 주제의 possession 체인 추적 방법론(분석 질문 6) 전용 하위 폴더입니다. 분석 기획은 [`../PLAN.md`](../PLAN.md), 구역 기반 전진 경로는 [`../zone_progression/`](../zone_progression/), 백로그 항목은 `ideas/backlog.md`의 "2024 유로 스페인의 빌드업 패턴"을 참고하세요.

## 방법론: 체인 정의와 시각화

`plot_possession_chain_progression()`(`src/visualizer.py`)의 계산 순서는 다음과 같습니다.

1. **체인 필터링**: `type == 'Pass'`, `team == possession_team == 'Spain'`, 성공(`pass_outcome` 결측) 조건으로 스페인의 소유 구간 성공 패스만 남긴 뒤, 같은 `possession` id 안에 이 조건을 만족하는 패스가 `min_chain_length`(기본 2) 회 이상인 possession만 "체인"으로 인정합니다. `possession`은 팀 구분 없이 부여되므로 `team`뿐 아니라 `possession_team`도 함께 걸러야 상대 팀 이벤트가 섞이지 않습니다.
2. **메인 패널(구역 점유 + 전진 화살표)**: `plot_zone_progression()`과 동일한 30구역 Juego de Posición 그리드·시각화 스타일을 재사용하되, 위에서 걸러낸 체인 패스만 집계합니다.
3. **대표 체인 미니 패널**: 체인의 마지막 패스가 도착한 구역의 y채널(Left Wide/HS, Central, Right Wide/HS)로 체인을 분류하고, 왼쪽/오른쪽 각각 패스 수가 가장 많은 체인 1개씩을 뽑아 실제 좌표(구역 중심이 아닌 원본 location) 그대로 순서대로 이어 그립니다.

**설계 변경**: 대표 체인을 처음엔 메인 히트맵 위에 겹쳐 그렸으나, 왔다갔다하는 패스가 많은 체인은 선이 피치를 가로질러 기존 화살표와 뒤섞여 못 알아볼 정도로 지저분해졌습니다. 그래서 메인 패널과 겹치지 않는 별도의 미니 피치 2개(오른쪽 열)로 분리했습니다.

**데이터 검토** (`scripts/review_possession_chains_data.py`, 7경기 전체): `possession` 결측 0건. 경기당 possession 개수는 144~215개, 스페인 성공 패스가 1개 이상 포함된 possession은 64~88개. possession당 성공 패스 수는 평균 5.7~11.4개, 중앙값 4~8개(최댓값 51, 조지아전) - 체인으로 다루기에 충분한 표본입니다. 최솟값은 모든 경기에서 1(패스 없이 소유권만 짧게 가진 possession)이라 `min_chain_length=2`로 걸러냅니다.

In [ ]:
import os
import sys

if sys.platform.startswith('win') and hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

sys.path.append(os.path.dirname(os.path.dirname(os.getcwd())))

import matplotlib.pyplot as plt
from src.data_loader import get_competition_matches, get_match_events
from src.visualizer import plot_possession_chain_progression

COMPETITION_ID = 55  # UEFA Euro
SEASON_ID = 282      # 2024
TEAM = "Spain"

output_dir = os.path.join(os.getcwd(), "processed", "spain_euro2024_possession_chains")
os.makedirs(output_dir, exist_ok=True)

In [ ]:
matches = get_competition_matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)
spain_matches = matches[(matches['home_team'] == TEAM) | (matches['away_team'] == TEAM)].copy()
spain_matches = spain_matches.sort_values('match_date')
spain_matches[['match_id', 'match_date', 'home_team', 'away_team', 'home_score', 'away_score', 'competition_stage']]

In [ ]:
summaries = {}

for _, match in spain_matches.iterrows():
    match_id = match['match_id']
    opponent = match['away_team'] if match['home_team'] == TEAM else match['home_team']
    stage = match['competition_stage']

    events = get_match_events(match_id=match_id)

    fig, axd, summary = plot_possession_chain_progression(
        events_df=events,
        team_name=TEAM,
        title=f"Spain Possession Chain Progression - {stage} vs {opponent}",
    )
    summaries[f"{stage} vs {opponent}"] = summary

    filename = f"{stage.lower().replace(' ', '_')}_vs_{opponent.lower().replace(' ', '_')}.png"
    out_path = os.path.join(output_dir, filename)
    fig.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='#1e1e1e')
    plt.show()
    plt.close(fig)
    print(f"저장 완료: {out_path}")

In [ ]:
import pandas as pd

rows = []
for match_label, summary in summaries.items():
    stats = summary.groupby('final_side')['n_passes'].agg(['count', 'mean', 'median'])
    for side in ['Left', 'Central', 'Right']:
        if side in stats.index:
            rows.append({
                'match': match_label, 'final_side': side,
                'n_chains': stats.loc[side, 'count'],
                'mean_passes': round(stats.loc[side, 'mean'], 2),
                'median_passes': stats.loc[side, 'median'],
            })
pd.DataFrame(rows)

## 관찰 기록

7경기 결과를 보며 경기별 차이와 대회 전체를 관통하는 패턴(왼쪽/오른쪽 진출 체인의 패스 수 차이 등)을 정리한 결과는 `RESULTS.md`에 문서화할 예정입니다 (아직 미작성).